In [1]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_paga"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_paga"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

adata1_cd8 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_cd8_subclustered_v2.h5ad")
print(f"Loaded: {adata1_cd8.n_obs} cells")
print("Neighbors already present:", "neighbors" in adata1_cd8.uns)
print(adata1_cd8.obs["cd8_subtype_v2"].value_counts())

Loaded: 9794 cells
Neighbors already present: True
cd8_subtype_v2
Activated CD8 T cells       4941
Naive/Memory CD8 T cells    2246
Effector CD8 T cells        1668
NK-like CD8 T cells          840
Cycling CD8 T cells           99
Name: count, dtype: int64


In [2]:
adata1_cd8.obs["cd8_subtype_v2"] = adata1_cd8.obs["cd8_subtype_v2"].astype(str).astype("category")

sc.tl.paga(adata1_cd8, groups="cd8_subtype_v2")

paga_connectivities = pd.DataFrame(
    adata1_cd8.uns["paga"]["connectivities"].toarray(),
    index=adata1_cd8.obs["cd8_subtype_v2"].cat.categories,
    columns=adata1_cd8.obs["cd8_subtype_v2"].cat.categories,
)
print("PAGA connectivity matrix (CD8 states):")
print(paga_connectivities.round(3))

fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.paga(adata1_cd8, color="cd8_subtype_v2", threshold=0.05,
           node_size_scale=2, edge_width_scale=1.5,
           fontsize=9, frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_CD8_paga_graph.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

paga_connectivities.to_csv(RESULTS_DIR / "GSE114725_CD8_paga_connectivities.csv")
print("Saved PAGA graph and connectivity matrix")

PAGA connectivity matrix (CD8 states):
                          Activated CD8 T cells  Cycling CD8 T cells  \
Activated CD8 T cells                     0.000                0.227   
Cycling CD8 T cells                       0.227                0.000   
Effector CD8 T cells                      0.386                0.990   
NK-like CD8 T cells                       0.370                0.430   
Naive/Memory CD8 T cells                  0.452                0.462   

                          Effector CD8 T cells  NK-like CD8 T cells  \
Activated CD8 T cells                    0.386                0.370   
Cycling CD8 T cells                      0.990                0.430   
Effector CD8 T cells                     0.000                0.279   
NK-like CD8 T cells                      0.279                0.000   
Naive/Memory CD8 T cells                 0.657                0.199   

                          Naive/Memory CD8 T cells  
Activated CD8 T cells                        0.4

In [3]:
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.paga(adata1_cd8, color="cd8_subtype_v2", threshold=0.05,
           node_size_scale=2, edge_width_scale=0.5,
           fontsize=9, frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_CD8_paga_graph.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved corrected PAGA graph")

Saved corrected PAGA graph


In [4]:
import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_paga"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_paga"

# ----------------------------
# Full T cell lineage PAGA — GSE114725: 3 CD4 states + 5 CD8 states +
# True NK + NKT = 10 nodes, the fine-resolution equivalent of the
# original T cell lineage PAGA (which only had 5, undivided, nodes).
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")

lineage_categories = [
    "CD4 Naive/Resting T cells", "CD4 Activated T cells", "Regulatory T cells (Tregs, CD4+)",
    "Activated CD8 T cells", "Naive/Memory CD8 T cells", "Effector CD8 T cells",
    "NK-like CD8 T cells", "Cycling CD8 T cells",
    "True NK cells", "NKT cells",
]
adata1_lineage = adata1[adata1.obs["cell_type_fine"].isin(lineage_categories)].copy()
print(f"Full lineage pool: {adata1_lineage.n_obs} cells")
print(adata1_lineage.obs["cell_type_fine"].value_counts())

# Proactive zero-variance check before HVG/scaling
X_check = adata1_lineage.raw.X if adata1_lineage.raw is not None else adata1_lineage.X
if hasattr(X_check, "toarray"): X_check = X_check.toarray()
variances = X_check.var(axis=0)
n_zero_var = (variances == 0).sum()
print(f"Zero-variance genes (full raw panel): {n_zero_var}")
if n_zero_var > 0:
    keep_genes = adata1_lineage.raw.var_names[variances != 0]
    adata1_lineage_raw_filtered = adata1_lineage.raw.to_adata()[:, keep_genes].copy()
    adata1_lineage.raw = adata1_lineage_raw_filtered
    common = [g for g in adata1_lineage.var_names if g in keep_genes]
    adata1_lineage = adata1_lineage[:, common].copy()

Full lineage pool: 31813 cells
cell_type_fine
CD4 Activated T cells               8532
CD4 Naive/Resting T cells           7555
Activated CD8 T cells               4941
Regulatory T cells (Tregs, CD4+)    2992
Naive/Memory CD8 T cells            2246
True NK cells                       2192
Effector CD8 T cells                1668
NK-like CD8 T cells                  840
NKT cells                            748
Cycling CD8 T cells                   99
Name: count, dtype: int64
Zero-variance genes (full raw panel): 122


In [5]:
sc.pp.highly_variable_genes(adata1_lineage, n_top_genes=2000, flavor="seurat")
adata1_lineage_hvg = adata1_lineage[:, adata1_lineage.var.highly_variable].copy()
sc.pp.scale(adata1_lineage_hvg, max_value=10)
sc.tl.pca(adata1_lineage_hvg, svd_solver="arpack", random_state=0)

sce.pp.harmony_integrate(adata1_lineage_hvg, key="patient", basis="X_pca", random_state=0)
sc.pp.neighbors(adata1_lineage_hvg, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)

adata1_lineage_hvg.obs["cell_type_fine"] = adata1_lineage.obs["cell_type_fine"].astype(str).astype("category").values

sc.tl.paga(adata1_lineage_hvg, groups="cell_type_fine")

paga_lineage = pd.DataFrame(
    adata1_lineage_hvg.uns["paga"]["connectivities"].toarray(),
    index=adata1_lineage_hvg.obs["cell_type_fine"].cat.categories,
    columns=adata1_lineage_hvg.obs["cell_type_fine"].cat.categories,
)
print("Full T cell lineage PAGA connectivity matrix:")
print(paga_lineage.round(3))

paga_lineage.to_csv(RESULTS_DIR / "GSE114725_full_lineage_paga_connectivities.csv")
print("\nSaved connectivity matrix")

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_highly_variable_genes.py:336: RuntimeWarning: invalid value encountered in log
  dispersion = np.log(dispersion)
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: `n_top_genes` > number of normalized dispersions, returning all genes with normalized dispersions.
  return fn(*args_all, **kw)
2026-07-22 12:38:22,865 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-22 12:38:42,334 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-22 12:38:42,883 - harmonypy - INFO - Iteration 1 of 10
2026-07-22 12:39:21,154 - harmonypy - INFO - Iteration 2 of 10
2026-07-22 12:40:01,476 - harmonypy - INFO - Converged after 2 iterations
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install

Full T cell lineage PAGA connectivity matrix:
                                  Activated CD8 T cells  \
Activated CD8 T cells                             0.000   
CD4 Activated T cells                             0.273   
CD4 Naive/Resting T cells                         0.010   
Cycling CD8 T cells                               0.537   
Effector CD8 T cells                              1.000   
NK-like CD8 T cells                               1.000   
NKT cells                                         0.081   
Naive/Memory CD8 T cells                          1.000   
Regulatory T cells (Tregs, CD4+)                  0.055   
True NK cells                                     0.031   

                                  CD4 Activated T cells  \
Activated CD8 T cells                             0.273   
CD4 Activated T cells                             0.000   
CD4 Naive/Resting T cells                         0.399   
Cycling CD8 T cells                               0.164   
Effector 